> `oeai_mod_bromcom_bronze_sql` - Bronze notebook for Bromcom + SQL connection

In [ ]:
%run oeai_mod_bromcom_env_var

In [ ]:
# Initialise Logging
oeai.notebook = SimpleNamespace(
    name = "oeai_mod_bromcom_bronze_sql.ipynb",
    buildversion = "20251105.1",
    buildtimestamp = "2025-11-05T16:00:00Z",
)
oeai.log.init()
oeai.log.start_block("Notebook Init")

In [ ]:
# HACF guard — raise hard if connection is not "sql"
if (connection or "").strip().lower() != "sql":
    oeai.log.critical(f"HACF: connection='{connection}' != 'sql'. Intentionally stopping execution.")
    raise RuntimeError(
        f"HACF: connection='{connection}' != 'sql'. Intentionally stopping execution."
    )
oeai.log.debug("Guard passed: connection == 'sql'. Proceeding…")

In [ ]:
###
# Explore available tables and views
###

tables_df = spark.read.jdbc(
    url=jdbc_url,
    table="INFORMATION_SCHEMA.TABLES",
    properties=db_properties
)

display(
    tables_df.select("TABLE_SCHEMA", "TABLE_NAME", "TABLE_TYPE")
)

In [ ]:
oeai.log.end_block(index=0, include_target=True)
oeai.log.checkpoint("Init")

In [ ]:
oeai.log.start_block("Processing")

In [ ]:
from pyspark.sql.functions import col
import re, os

# Helper to make safe folder names (avoid spaces/brackets/etc.)
def safe(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]", "_", s)

# (Optional) tweak JDBC fetch size for bigger pulls
db_props = dict(db_properties)
db_props["fetchsize"] = "10000"

# === 1) Parse the list into (schema, viewName, originalName) tuples ===
parsed = []
for name in selected_views:
    if "_" not in name:
        # print(f"⚠️ Skipping '{name}' (expected '<schema>_<view>')")
        oeai.log.warning(f"Skipping '{name}' (expected '<schema>_<view>')", name=name, schema=schema, view=view)
        continue
    schema, view = name.split("_", 1)  # split only on first underscore
    parsed.append((schema, view, name))

total = len(parsed)
results = []

# === 2) Loop and materialize to Delta ===
for i, (schema, view, original_name) in enumerate(parsed, start=1):
    fq = f"[{schema}].[{view}]"
    # target_path = f"{bronze_path}/{safe(original_name)}"  # flat, includes schema in name
    target_path = os.path.join(bronze_path, safe(original_name))  # flat, includes schema in name

    # print(f"[{i}/{total}] Reading view {schema}.{view} …")
    oeai.log.debug(f"[{i}/{total}] Reading view {schema}.{view} …", schema=schema, view=view)
    try:
        df = spark.read.jdbc(
            url=jdbc_url,
            table=f"(SELECT * FROM {fq}) AS t",
            properties=db_props
        )

        # print(f"[{i}/{total}] Writing Delta -> {target_path}")
        oeai.log.debug(f"[{i}/{total}] Writing Delta -> {target_path}", schema=schema, view=view, target_path=target_path)
        (df.write
            .format("delta")
            .mode("overwrite")                 # or 'append' if preferred
            .option("overwriteSchema", "true")
            .save(target_path))

        status = "ok"
        # print(f"[{i}/{total}] ✅ Done: {schema}.{view}")
        oeai.log.info(f"[{i}/{total}] Done: {schema}.{view}", schema=schema, view=view)
    except Exception as e:
        status = f"failed: {str(e)[:200]}"
        # print(f"[{i}/{total}] ❌ Failed: {schema}.{view} -> {status}")
        oeai.log.exception(f"[{i}/{total}] ❌ Failed: {schema}.{view} -> {status}", schema=schema, view=view, status=status)

    results.append((schema, view, original_name, target_path, status))

# === 3) Summary table ===
result_df = spark.createDataFrame(results, ["schema", "view", "name_used_for_folder", "path", "status"])
display(result_df)


In [ ]:
oeai.log.end_block(index=0, include_target=True)
oeai.log.shutdown()
oeai.log.checkpoint("shutdown")